![ngo_project_image](ngo_project_image.jpg)

GoodThought NGO has been a catalyst for positive change, focusing its efforts on education, healthcare, and sustainable development to make a significant difference in communities worldwide. With this mission, GoodThought has orchestrated an array of assignments aimed at uplifting underprivileged populations and fostering long-term growth.

This project offers a hands-on opportunity to explore how data-driven insights can direct and enhance these humanitarian efforts. In this project, you'll engage with the GoodThought PostgreSQL database, which encapsulates detailed records of assignments, funding, impacts, and donor activities from 2010 to 2023. This comprehensive dataset includes:

- **`Assignments`:** Details about each project, including its name, duration (start and end dates), budget, geographical region, and the impact score.
- **`Donations`:** Records of financial contributions, linked to specific donors and assignments, highlighting how financial support is allocated and utilized.
- **`Donors`:** Information on individuals and organizations that fund GoodThought’s projects, including donor types.

Refer to the below ERD diagram for a visual representation of the relationships between these data tables:
<img src="erd.png" alt="ERD" width="50%" height="50%">


You will execute SQL queries to answer two questions, as listed in the instructions. Good luck!


In [13]:
SELECT *
FROM Assignments as A
LEFT JOIN Donations as D
ON A.ASSIGNMENT_ID = D.ASSIGNMENT_ID
LEFT JOIN DONORS as DON
ON D.DONOR_ID = DON.DONOR_ID

In [14]:
SELECT A.assignment_name,
	A.region,
	ROUND(SUM(D.amount),2) AS num_total_donations,
	DON.donor_type
FROM Assignments as A
LEFT JOIN Donations as D
ON A.ASSIGNMENT_ID = D.ASSIGNMENT_ID
LEFT JOIN DONORS as DON
ON D.DONOR_ID = DON.DONOR_ID
GROUP BY a.region, a.assignment_name, don.donor_type

In [15]:
-- highest_donation_assignments

SELECT A.assignment_name,
	A.region,
	ROUND(SUM(D.amount),2) AS rounded_total_donation_amount,
	DON.donor_type
FROM Assignments as A
LEFT JOIN Donations as D
ON A.ASSIGNMENT_ID = D.ASSIGNMENT_ID
LEFT JOIN DONORS as DON
ON D.DONOR_ID = DON.DONOR_ID
GROUP BY a.region, a.assignment_name, don.donor_type
HAVING ROUND(SUM(D.amount),2) <> 0
ORDER BY rounded_total_donation_amount DESC
LIMIT 5;


,assignment_name,region,rounded_total_donation_amount,donor_type
0,Assignment_3033,East,3840.66,Individual
1,Assignment_300,West,3133.98,Organization
2,Assignment_4114,North,2778.57,Organization
3,Assignment_1765,West,2626.98,Organization
4,Assignment_268,East,2488.69,Individual


In [16]:
SELECT 
	A.assignment_id,
	count(d.donation_id)
FROM Assignments as A
LEFT JOIN Donations as D
ON A.ASSIGNMENT_ID = D.ASSIGNMENT_ID
GROUP BY a.assignment_id

In [17]:
-- top_regional_impact_assignments
WITH FIRSTCTE AS (
	SELECT 
	A.assignment_id,
	A.assignment_name,
	A.region,
	A.impact_score,
	count(d.donation_id) as num_total_donations
FROM assignments as A
LEFT JOIN donations as D
ON A.assignment_id = D.assignment_id
GROUP BY A.assignment_id, A.assignment_name, A.region, A.impact_score
),
SECONDCTE AS (
	SELECT 
	assignment_id,
	assignment_name,
	region,
	impact_score,
	ROW_NUMBER() OVER (PARTITION BY region ORDER BY impact_score DESC) AS rn
FROM assignments
	)
	
SELECT 
	f.assignment_name, 
	f.region, 
	f.impact_score, 
	f.num_total_donations
FROM FIRSTCTE as f
JOIN SECONDCTE as s
ON f.assignment_id = s.assignment_id
WHERE s.rn = 1
ORDER BY f.region, f.impact_score DESC;


,assignment_name,region,impact_score,num_total_donations
0,Assignment_316,East,10.00,2
1,Assignment_2253,North,9.99,1
2,Assignment_3547,South,10.00,1
3,Assignment_3764,West,9.99,1


In [18]:
SELECT assignment_id,
	region,
	impact_score,
	ROW_NUMBER() OVER (PARTITION BY region ORDER BY impact_score DESC) AS SCORE
FROM assignments
	order by impact_score desc
